# 04 - Token-Based Splitting

## Purpose

This notebook splits the cleaned PDF documents into token-sized chunks.

Version 1 used character-based chunking. Version 2 uses token-based splitting, which is more aligned with how language models process text.

The output of this notebook will be used for OpenAI embeddings and Chroma vector indexing.

## Main Changes in Version 2

This version adds:

- Token-aware splitting
- 500-token chunks
- 50-token overlap
- Metadata preservation from earlier document processing
- Better preparation for LLM-based retrieval

## Input

This notebook recreates the formatted document workflow from earlier steps.

## Main Output

- `docs_list_tokens_split`

## Notebook Flow

```text
Formatted documents
        ↓
TokenTextSplitter
        ↓
Token-sized chunks
        ↓
Preserve metadata
        ↓
Inspect chunk previews
        ↓
Prepare for embeddings and vector storage

In [0]:
%pip install langchain langchain-community langchain-openai pypdf tiktoken chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
import os

os.environ["OPENAI_API_KEY"] = "Open AI API Key"

In [0]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
import time

pdf_path = "/Volumes/workspace/365pdf/365pdfupload/Introduction_to_Tableau.pdf"

loader_pdf = PyPDFLoader(pdf_path)
docs_list = loader_pdf.load()

markdown_transcript = ""

for doc in docs_list:
    page_number = doc.metadata.get("page", "unknown")
    page_text = doc.page_content

    markdown_transcript += f"\n\n# Introduction to Tableau\n"
    markdown_transcript += f"## Page {page_number}\n\n"
    markdown_transcript += page_text

headers_to_split_on = [
    ("#", "Section Title"),
    ("##", "Page Title")
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

docs_list_md_split = md_splitter.split_text(markdown_transcript)

chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

formatting_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a document-cleaning assistant.

Your task is to clean text extracted from a PDF.

Rules:
- Fix broken line breaks and awkward spacing.
- Improve punctuation and capitalization where needed.
- Keep the original meaning unchanged.
- Do not add new information.
- Do not summarize.
- Do not remove important details.
- Return only the cleaned text.
"""
    ),
    (
        "human",
        """
Clean and format the following extracted PDF text:

{text}
"""
    )
])

formatting_chain = formatting_prompt | chat | StrOutputParser()

formatted_docs_list = []

for i, doc in enumerate(docs_list_md_split):
    print(f"Formatting document {i + 1} of {len(docs_list_md_split)}")

    text_to_format = doc.page_content[:3000]

    formatted_text = formatting_chain.invoke({
        "text": text_to_format
    })

    formatted_doc = Document(
        page_content=formatted_text,
        metadata=doc.metadata
    )

    formatted_docs_list.append(formatted_doc)

    time.sleep(0.5)

print("Formatted documents ready:", len(formatted_docs_list))

In [0]:
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50
)

docs_list_tokens_split = token_splitter.split_documents(formatted_docs_list)

print("Token-split documents created:", len(docs_list_tokens_split))

In [0]:
for i, doc in enumerate(docs_list_tokens_split[:5]):
    print(f"\nToken Document {i}")
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.page_content[:800])
    print("-" * 80)

In [0]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

for i, doc in enumerate(docs_list_tokens_split[:5]):
    token_count = len(encoding.encode(doc.page_content))
    print(f"Document {i} token count:", token_count)

In [0]:
total_tokens = sum(
    len(encoding.encode(doc.page_content))
    for doc in docs_list_tokens_split
)

print("Number of token chunks:", len(docs_list_tokens_split))
print("Approximate total tokens:", total_tokens)
print("Average tokens per chunk:", total_tokens / len(docs_list_tokens_split))